# 1. 上下文增强

## 本节解决什么问题

基础 RAG 的固定 chunk 往往能命中相关内容，但不一定把回答所需的上下文带完整。观察 Baseline 的真实输出会发现：

- 命中片段本身太碎，关键定义落在 chunk 边界外；
- 上下支撑句被截断，推导依据读不完整；
- 同一段内的补充信息分散在不同 chunk 里，只带回了一半。

因此，baseline 的问题通常不是"完全没检到"，而是**检到了，但上下文不够完整**。本节要做的不是"再想办法检更多"，而是**在检索已经大致命中的前提下，把更适合生成的上下文补回来**。

本节围绕这一核心问题介绍三种方法：

- **Sentence Window（句窗）**：命中句子后补回前后邻句，补回局部连续解释。
- **Small-to-Big（父子块）**：用小块精准定位，再回填父块生成，补回同段完整信息。
- **AutoMerging（自动合并）**：当同一父块下多个子块被命中时，按命中密度自动合并，补回分散证据的完整上下文。

为了让教程既可信又好读，本节采用两层评测：
1. **自然主集**：沿用同一套 `QA_INDICES`（当前 **24** 题），观察整体趋势、提分题数和上下文成本。
2. **方法级面板**：为每种增强方法展示代表题，说明 baseline 缺了什么、增强方法补回了什么。

> 小样本上 0/1/2 裁判有随机性，结论看**方向、逐题表、优势样例和上下文证据链**，不必抠小数点第二位。

## 本节与其他增强层的边界

进入具体方法前，先明确三层增强的分工：

| 章节 | 解决什么问题 | 不解决什么问题 |
|---|---|---|
| **6.1 上下文增强（本节）** | 检索已大致命中，但送入生成的上下文还不够完整 | 不讲 query 改写、不讲多轮流程控制、不讲 agent 编排 |
| **6.2 流程增强** | 一次检索流程不够，需要迭代、递归、路由、纠错或自评 | 不讲跨轮状态、不讲多文档 agent 编排 |
| **6.3 系统增强** | 问题跨轮、跨文档、跨工具或依赖系统状态 | 不是本节范围 |

简言之：**本节只讲"检索命中后，如何恢复更完整上下文"**。如果你发现问题的根源不是"命中了但上下文不够"，而是"一次流程本身不够"或"需要跨轮/跨文档推理"，请继续学习后续章节。

## 环境准备与统一实验设置

本节使用智谱 AI 的 `GLM-4-Flash` 做生成模型，使用本地 `BAAI/bge-small-zh-v1.5` 做 embedding。

运行前请确保：
1. 安装依赖：`modelscope`、`sentence-transformers`、`torch`
2. 在项目根目录的 `.env` 文件中配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载 embedding 模型到 `./models/`

> **数据说明**：使用与「3. 索引阶段」相同的数据集（南瓜书《机器学习公式详解》）。

**统一实验协议**（三种方法的共同对照）：

| 项目 | 设置 |
|---|---|
| 数据 | `../3. 索引阶段/data/pumpkin_book.pdf` |
| 问答 | `../3. 索引阶段/data/train_dataset.json` |
| 题集 | `QA_INDICES` = 24 题（基础 20 + 4 道加难） |
| 生成模型 | `glm-4-flash-250414`，temperature=0 |
| 向量模型 | 本地 `BAAI/bge-small-zh-v1.5` |
| Baseline | 字符块 **256 / overlap 20 / k=4** |
| 上下文预算 | `CONTEXT_CHAR_BUDGET=2500`，各路径统一截断 |
| 评分 | LLM 裁判 0/1/2 分，优先看方向与逐题差异 |

三种增强路径各自独立检索，仅 `CONTEXT_CHAR_BUDGET` 在进入生成前统一截断，确保公平对比。

In [1]:
import os
import re
import warnings
import pandas as pd
from IPython.display import Markdown, display
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import sys
sys.path.insert(0, ".")
from _common import (
    get_cleaned_pdf_documents, open_or_build_chroma,
    trim_context_to_budget, load_qna_subset, CONTEXT_CHAR_BUDGET,
    QA_PATH, answer_from_context_fn, run_shared_eval, build_compare_table,
)
from _context_enhance_utils import (
    score_summary, save_context_enhance_outputs,
    method_diff_summary, build_method_evidence_index,
    display_baseline_panel, display_sentence_window_panel,
    display_small_to_big_panel, display_auto_merging_panel,
)
def split_sentences_for_window(text: str, max_piece: int = 320):
    raw = [s.strip() for s in re.split(r"(?<=[。！？!?])", text) if s.strip()]
    out = []
    for s in raw:
        if len(s) <= max_piece:
            out.append(s)
        else:
            for i in range(0, len(s), max_piece):
                out.append(s[i : i + max_piece])
    return out
warnings.filterwarnings("ignore")
print("✅ 公共底座与 6.1 展示辅助已引入")


✅ 公共底座与 6.1 展示辅助已引入


/usr/local/Caskroom/miniconda/base/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**本节专用辅助：检索 / 命中解析 / 上下文预览**

这里编写一些函数工具，让我们更聚焦上下文的效果。调用 retriever 接口、从命中 Document 里抽 `sentence_id` / `child_id`、按字符上限切定长 chunk、以及预览最终送进 LLM 的上下文。


In [2]:
def retriever_hits(retriever, question: str):
    """Runnable 检索接口：invoke 返回 Document 列表。"""
    return retriever.invoke(question)
def hit_sentence_ids_from_docs(docs):
    out = []
    for d in docs:
        sid = d.metadata.get("sentence_id")
        if sid is None:
            print("  [提示] 命中缺少 sentence_id，已跳过。若 persist 目录刚升版，请删旧库后重建索引。")
            continue
        try:
            out.append(int(sid))
        except (TypeError, ValueError):
            print(f"  [提示] 无效 sentence_id={sid!r}，已跳过。")
    # 去重但保留首次命中顺序，避免后续窗口/父块拼接顺序抖动。
    # 去重但保留首次命中顺序，避免同一题多次运行展示顺序不稳定。
    seen, uniq = set(), []
    for x in out:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq
def hit_child_ids_from_docs(docs):
    out = []
    for d in docs:
        cid = d.metadata.get("child_id")
        if cid is None:
            print("  [提示] 命中缺少 child_id，已跳过。若 persist 目录刚升版，请删旧库后重建索引。")
            continue
        try:
            out.append(int(cid))
        except (TypeError, ValueError):
            print(f"  [提示] 无效 child_id={cid!r}，已跳过。")
    seen, uniq = set(), []
    for x in out:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq
def load_chunks(chunk_size=256, chunk_overlap=20):
    docs = get_cleaned_pdf_documents()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        keep_separator=True,
    )
    return splitter.split_documents(docs)
def build_retriever(chunk_size=256, chunk_overlap=20, k=4):
    persist_dir = f"./chroma_db/baseline_{chunk_size}_{chunk_overlap}"
    chunks = load_chunks(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    ids = [f"b{i}" for i in range(len(chunks))]
    vs = open_or_build_chroma(persist_dir, chunks, ids)
    # 统一 k 口径，确保方法间差异主要来自“补上下文策略”而非召回数量。
    return vs.as_retriever(search_kwargs={"k": k})


**再把评测题集固定下来**

为了让 Baseline、句窗、Small-to-Big 和 AutoMerging 的对比尽量公平，这里用 `_common.load_qna_subset` 从 `train_dataset.json` 中按 `QA_INDICES` 取出共用题集。`EVAL_MAX_UNIQ_QUESTIONS` 环境变量可以临时裁剪题集（用于 nbconvert 快速验证）。


In [3]:
# 20 道基础 + 4 道加难（与 train_dataset 下标一致，便于后文按「末四题」作困难子集）
HARD_QA_INDICES = [21, 41, 55, 30]
QA_INDICES = list(range(20)) + HARD_QA_INDICES
qna_dict = load_qna_subset(QA_PATH, QA_INDICES)

# 测试 / CI 模式：通过环境变量裁剪题集，避免 nbconvert 长时间评估
_lim = os.environ.get("EVAL_MAX_UNIQ_QUESTIONS", "").strip()
if _lim:
    qna_dict = dict(list(qna_dict.items())[: max(0, int(_lim))])
    print(f"⚠️ 测试模式：仅使用前 {len(qna_dict)} 道题（EVAL_MAX_UNIQ_QUESTIONS）")

print(f"✅ 本节共用 {len(qna_dict)} 题（QA_INDICES={QA_INDICES[:5]}...）")


✅ 本节共用 24 题（QA_INDICES=[0, 1, 2, 3, 4]...）


**评估接口（在 `_common.py`）**

`simple_eval_2pt` / `answer_from_context_fn` / `run_shared_eval` / `build_compare_table` 的完整定义在公共底座，**本节不再内联粘贴**，避免与 6.2/6.3/评估专章三份重复。下面直接调用即可。


## Baseline 观察：固定 chunk 对照

本节只保留 **1 个固定 baseline**，作为三种方法的共同对照。baseline 的职责是建立一个可复现、容易理解的起点，而不是故意做弱。

**Baseline 配置**：字符块 256 / overlap 20 / k=4。

先不做任何上下文增强，只用基础向量检索回答 `qna_dict`。我们关注两个观察点：

1. baseline 是否"完全没检到"？
2. 还是"检到了，但上下文不够完整"？

In [4]:
# Baseline：固定长度分块 + 向量 top-k（作为本节可复现的固定 chunk 对照）
baseline_retriever = build_retriever(chunk_size=256, chunk_overlap=20, k=4)
def baseline_hits(question: str):
    return retriever_hits(baseline_retriever, question)
def baseline_expanded_join(docs) -> str:
    """top-k 块按检索顺序拼接（未做 CONTEXT_CHAR_BUDGET）。"""
    return "\n\n".join(d.page_content for d in docs)
def baseline_context_from_docs(docs) -> str:
    # Baseline 也走统一 budget，保证与各增强路径在同一上下文上限下对比。
    return trim_context_to_budget(baseline_expanded_join(docs), CONTEXT_CHAR_BUDGET)
def baseline_context(question: str) -> str:
    return baseline_context_from_docs(baseline_hits(question))
baseline_answer = answer_from_context_fn(baseline_context)
baseline_df = run_shared_eval(baseline_answer, qna_dict)
print(f"Baseline 评测完成：{len(baseline_df)} 题。详细逐题结果在后文分析面板与总览表展示。")


  -> 加载已有索引: ./chroma_db/baseline_256_20


Baseline 评测完成：24 题。详细逐题结果在后文分析面板与总览表展示。


### Baseline 真实检索观察

下面选一题 Baseline 真实未得分的样例，直接看 top-k 片段预览和送入 LLM 的完整上下文。baseline 的典型模式是：检索结果相关，但固定 chunk 从半句开始、在关键解释处截断，或只覆盖局部证据。

In [5]:
display_baseline_panel(
    baseline_df=baseline_df,
    baseline_hits=baseline_hits,
    baseline_context_from_docs=baseline_context_from_docs,
)


### Baseline 真实检索观察

**代表题**：根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差异。请解释在这种情况下，如何根据不同的机器学习算法（如一元线性回归和多项式回归）评估它们的相对优劣，并请用式(1.1)到式(1.5)中的概念来支撑你的解释。

**Baseline 得分**：0

Baseline 的作用是建立对照：它经常能命中相关片段，但固定 chunk 可能从半句开始、在关键解释处截断，或只覆盖局部证据。

,top-k,片段预览,字符数
0,1,，因此并不能简单用比较大小来直接判定算法（或者模型）之间的优劣，而需要更置信的方法来进行判定...,256
1,2,机器学习算法之间没有绝对的优劣之分，只有是否适合当前待解决的问题之分，例如上述测试集中的数据...,256
2,3,问题复杂化，假设学校数量和房价呈y=wx2+b一元二次函数关系，此时问题变为了线性回归中的多...,256
3,4,习算法有不同的偏好，我们称为“归纳偏好”。对于当前房价预测这个例子来说，这两个算法学得的模型...,256


<details>
<summary>查看 Baseline 送入 LLM 的完整上下文</summary>

，因此并不能简单用比较大小来直接判定算法（或者模型）之间的优劣，而需要更置信的方法来进行判定。在此说明一下，如果不做算法理论研究，也不需要对算法（或模型）之间的优劣给出严谨的数学分析，本节可以暂时跳过。本节主要使用的数学知识是“统计假设检验”，该知识点在各个高校的概率论与数理统计教材（例如参考文献[1]）上均有讲解。此外，有关检验变量的公式，例如式(2.30)至式(2.36)，并不需要清楚是怎么来的（这是统计学家要做的事情），只需要会用即可。2.4.1式(2.26)的解释理解本公式时需要明确的是：ϵ是未知的

机器学习算法之间没有绝对的优劣之分，只有是否适合当前待解决的问题之分，例如上述测试集中的数据如果改为(年份:2022年;学校数量:3所;房价:9万/m2)则结论便逆转为多项式回归算法优于一元线性回归算法。1.4.1式(1.1)和式(1.2)的解释XfEote(La|X,f)=XfXhXx∈X−XP(x)I(h(x)̸=f(x))P(h|X,La)1⃝=Xx∈X−XP(x)XhP(h|X,La)XfI(h(x)̸=f(x))2⃝=Xx∈X−XP(x)XhP(h|X,La)122|X|3⃝=122|X|Xx∈X

问题复杂化，假设学校数量和房价呈y=wx2+b一元二次函数关系，此时问题变为了线性回归中的多项式回归问题，按照多项式回归算法可学得模型为y=x2。因此，以表1-1中数据作为训练集可以有多个假设空间，且在不同的假设空间中都有可能学得能够拟合训练集的模型，我们将所有能够拟合训练集的模型构成的集合称为“版本空间”。1.4归纳偏好在上一节“房价预测”的例子中，当选用一元线性回归算法时，学得的模型是一元一次函数，当选用多项式回归算法时，学得的模型是一元二次函数，所以不同的机器学习算法有不同的偏好，我们称为“归纳偏好”

习算法有不同的偏好，我们称为“归纳偏好”。对于当前房价预测这个例子来说，这两个算法学得的模型哪个更好呢？著名的“奥卡姆剃刀”原则认为“若有多个假设与观察一致，则选最简单的那个”，但是何为“简单”便见仁见智了，如果认为函数的幂次越低越简单，则此时一元线性回归算法更好，如果认为幂次越高越简单，则此时多项式回归算法更好，因此该方法其实并不“简单”，所以并不常用，而最常用的方法则是基于模型在测试集上的表现来评判模型之间的优劣。测试集是指由训练集之外的样本构成的集合，例如在当前房价预测问题中，通常会额外留有部分未参与

</details>

### Baseline 的典型上下文缺口

Baseline 使用 **256 字块、overlap=20、k=4**。本节用一个可复现的入门配置示范：检索结果可以相关，但回答所需的支撑上下文未必完整。

观察真实输出，baseline 的上下文缺口可分为三类，恰好对应本节三种增强方法：

| 上下文缺口类型 | 典型表现 | 对应增强方法 |
|---|---|---|
| 命中句相关，但前后支撑句不完整 | 关键解释落在 chunk 边界外 | **Sentence Window** |
| 小块定位准确，但回答需要同段完整解释 | 定义、条件、例子分散在段落中 | **Small-to-Big** |
| 多个证据分散在同一父块内 | 多个子证据各自只带回碎片 | **AutoMerging** |

后续三节都会遵循同一节奏：**先看方法如何补上下文，再看自然主集结果，最后用代表 case 说明它在哪类题上优于 baseline**。

### 本节通用读法

- **不要只看** LLM 最终回答（可能脑补）；更要看各方法小节里的**真实检索/补充内容**，以及**文末同题对比表**。
- **0～2 分**受裁判口径与生成随机性影响；讨论机制时优先看 **相对 Baseline 的提分题**、**平均分**，并对照各方法面板末尾「送入 LLM 的完整上下文」。
- **公平对比**：各增强路径**独立检索**，仅 **`CONTEXT_CHAR_BUDGET`** 在进入生成前统一截断。

## Sentence Window（句子窗口检索）

### 1. 方法直觉

Sentence Window 是最直觉的上下文补全方式：检索命中了一个句子，但答案所需的前后解释恰好落在 chunk 边界外。与其把整段都送给 LLM，不如只补回命中句的局部邻域——前后各 N 句，刚好构成一个完整的解释闭环。

### 2. 机制解释

检索命中后做了什么补全动作：

1. **索引时**：把文档按句子切分，每个句子单独嵌入，同时在元数据中记录该句子的前后邻居下标。
2. **检索时**：先用句子级 embedding 做检索，命中后不直接把这一句送给 LLM，而是查邻居映射，把左右各 `WINDOW_SIZE` 句拼上，再送给 LLM。

与 Baseline 的核心区别：

| 阶段 | Baseline | Sentence Window |
|---|---|---|
| 索引粒度 | 固定字符块 | 句子级 |
| 检索对象 | 整个 chunk | 单个句子 |
| 返回内容 | 命中的 chunk | 命中句子 + 前后窗口 |

### 3. 适用缺口

Sentence Window 更适合以下题型：

- 命中句前后有连续解释，但被 chunk 边界截断；
- 答案依赖的推导步骤、定义或例子就在命中句附近；
- 文档是连续叙述文本，句间逻辑紧密。

局限：如果证据散落在不同段落（不在邻句范围内），Sentence Window 无法补回，需要 Small-to-Big 或 AutoMerging。

### 4. 最小实现

In [6]:
base_docs = get_cleaned_pdf_documents()
# 全书按页顺序拼成一条长文本，再分句；句序即 neighbor_map 中的前后邻关系
full_text = "\n".join(d.page_content for d in base_docs)
sentences = split_sentences_for_window(full_text)
print(f"总句子数: {len(sentences)}")
sentence_map = {i: s for i, s in enumerate(sentences)}
WINDOW_SIZE = 2
# 句 i 的前后各 WINDOW_SIZE 句的全局下标（含 i）
neighbor_map = {
    i: list(range(max(0, i - WINDOW_SIZE), min(len(sentences), i + WINDOW_SIZE + 1)))
    for i in sentence_map
}
persist_dir_sw = "./chroma_db/sentence_window"
sentence_docs = [
    Document(page_content=s, metadata={"sentence_id": str(i)})
    for i, s in sentence_map.items()
]
sentence_vs = open_or_build_chroma(
    persist_dir_sw,
    sentence_docs,
    [str(i) for i in sentence_map.keys()],
)
sentence_retriever = sentence_vs.as_retriever(search_kwargs={"k": 4})
def sentence_window_hit_ids(question: str):
    return hit_sentence_ids_from_docs(retriever_hits(sentence_retriever, question))
def sentence_window_expanded_join(hit_ids) -> str:
    """多命中句的窗口取并集，去重后按句序拼接（未做 CONTEXT_CHAR_BUDGET）。"""
    # 用并集避免多命中窗口重复堆叠，保持 token 成本可控。
    window_ids = sorted({nid for hid in hit_ids for nid in neighbor_map.get(hid, [hid])})
    return "\n".join(sentence_map[i] for i in window_ids)
def sentence_window_context(question: str) -> str:
    return trim_context_to_budget(
        sentence_window_expanded_join(sentence_window_hit_ids(question)),
        CONTEXT_CHAR_BUDGET,
    )
sentence_window_answer = answer_from_context_fn(sentence_window_context)
sentence_window_df = run_shared_eval(sentence_window_answer, qna_dict)
print(f"Sentence Window 评测完成：{len(sentence_window_df)} 题。详细逐题结果在本方法分析面板展示。")


总句子数: 1812
  -> 加载已有索引: ./chroma_db/sentence_window


Sentence Window 评测完成：24 题。详细逐题结果在本方法分析面板展示。


### 5. 实验观察

下面展示 Sentence Window 在本轮 24 题里的真实整体结果：先对比 baseline 的逐题分数变化，再选一个真实提分题，展示它具体补回了哪些邻句。

In [7]:
display_sentence_window_panel(
    baseline_df=baseline_df,
    sentence_window_df=sentence_window_df,
    sentence_window_hit_ids=sentence_window_hit_ids,
    sentence_window_context=sentence_window_context,
    sentence_map=sentence_map,
    neighbor_map=neighbor_map,
    window_size=WINDOW_SIZE,
    build_compare_table=build_compare_table,
)


### Sentence Window 本轮真实结果

**分析结论**：本轮从 Baseline 的固定 chunk 改为句级命中后，主要收益来自补回命中句前后的连续解释。

| 指标 | 真实结果 |
|---|---:|
| Baseline 总分 | 18/48 |
| Sentence Window 总分 | 21/48 |
| 总分变化 | +3 |
| 严格提分题 | 4, 7, 9, 15, 22 |
| 边界样例 | 6, 14 |
| 持平题数 | 17 |

下面只展开发生分数变化的真实题目；具体补回内容在代表题里查看。

,题号,问题摘要,baseline,sentence_window,分数变化,状态
4,4,根据上述提供的上下文信息，请解释多元线性回归中的最小二乘法估计的数学表达式 \( \...,0,1,0 → 1,提分
6,6,"根据所提供的上下文信息，假设有一个二项分布B(m,p)的试验，其中事件发生次数为X。...",1,0,1 → 0,边界样例
7,7,根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差...,0,1,0 → 1,提分
9,9,根据提供的上下文信息，假设式(2.41)是关于某个学习算法的期望风险的分解形式。请解...,0,1,0 → 1,提分
14,14,根据给定的上下文信息，推导出式(5.14)中∆vih的计算表达式，并解释每一项的含义...,1,0,1 → 0,边界样例
15,15,解释8.4.5节中的式(8.25)和8.4.6节中的式(8.26)的区别，并给出一个...,0,1,0 → 1,提分
22,22,请根据给定的式(3.7)的向量化过程，描述如何使用Python中的NumPy库来实现...,0,1,0 → 1,提分


### Sentence Window 代表题：真实补回了哪些邻句？

**题号**：4  
**问题**：根据上述提供的上下文信息，请解释多元线性回归中的最小二乘法估计的数学表达式 \( \hat{w}^* \) 是如何通过向量内积的形式来简化的？并使用矩阵微分推导出 \( \hat{w}^* \) 对误差函数 \( E\hat{w} \) 的梯度。

**分数变化**：Baseline 0 → Sentence Window 1

这张表直接展示句窗检索的真实增量：左边是向量检索命中的句子，右边是同一次检索后按 `WINDOW_SIZE=2` 补回的邻句。

,命中序号,sentence_id,原始命中句,窗口额外补回
0,1,344,x2−¯x;...;xm−¯x)为去均值后的x；y=(y1;y2;...;ym)，yd=(y...,下面我们就尝试将上式进行向量化。 / 将1m(Pmi=1xi)2=¯xPmi=1xi代入分母...
1,2,387,除了需要解决如何找到xt+1以外，梯度下降法通常还需要解决如何判断当前点是否使得函数取到了最...,观察上式易知，此时∥∇f(xt)∥是固定常量，∥dt∥=1，所以当a也固定时，取θt=π，即...
2,3,559,根据该公式，可将要求解的极小化问题进一步简化为minwL(w)=minwXxi∈M(ˆyi−...,感知机学习算法：感知机模型的学习问题可以转化为求解损失函数的最优化问题，具体地，给定数据集T...
3,4,378,梯度下降法是一种迭代求解算法，其基本思路如下：先在定义域中随机选取一个点x0，将其代入函数f...,3.3.2梯度下降法不同于式(3.7)可求得闭式解，式(3.27)中的β没有闭式解，因此需要...


<details>
<summary>查看 Sentence Window 送入 LLM 的完整上下文</summary>

下面我们就尝试将上式进行向量化。
将1m(Pmi=1xi)2=¯xPmi=1xi代入分母可得w=Pmi=1yi(xi−¯x)Pmi=1x2i−¯xPmi=1xi=Pmi=1(yixi−yi¯x)Pmi=1(x2i−xi¯x)又因为¯yPmi=1xi=¯xPmi=1yi=Pmi=1¯yxi=Pmi=1¯xyi=m¯x¯y=Pmi=1¯x¯y且Pmi=1xi¯x=¯xPmi=1xi=¯x·m·1m·Pmi=1xi=m¯x2=Pmi=1¯x2，则有w=Pmi=1(yixi−yi¯x−xi¯y+¯x¯y)Pmi=1(x2i−xi¯x−xi¯x+¯x2)=Pmi=1(xi−¯x)(yi−¯y)Pmi=1(xi−¯x)2若令x=(x1;x2;...;xm)，xd=(x1−¯x;
x2−¯x;...;xm−¯x)为去均值后的x；y=(y1;y2;...;ym)，yd=(y1−¯y;y2−¯y;...;ym−¯y)为去均值后的y，（x、xd、y、yd均为m行1列的列向量）代入上式可得w=xTdydxTdxd
3.2.6式(3.9)的推导式(3.4)是最小二乘法运用在一元线性回归上的情形，那么对于多元线性回归来说，我们可以类似得到(w∗,b∗)=argmin(w,b)mXi=1(f(xi)−yi)2=argmin(w,b)mXi=1(yi−f(xi))2=argmin(w,b)mXi=1 yi− wTxi+b2为便于讨论，我们令ˆw=(w;b)=(w1;...;wd;b)∈R(d+1)×1,ˆxi=(xi1;
...;xid;1)∈R(d+1)×1，那么上式可以简化为ˆw∗=argminˆwmXi=1yi−ˆwTˆxi2=argminˆwmXi=1yi−ˆxTiˆw2根据向量内积的定义可知，上式可以写成如下向量内积的形式ˆw∗=argminˆwhy1−ˆxT1ˆw···ym−ˆxTmˆwiy1−ˆxT1ˆw...ym−ˆxTmˆw其中y1−ˆxT1ˆw...ym−ˆxTmˆw=y1...ym−ˆxT1ˆw...ˆxTmˆw=y−ˆxT1...ˆxTm·ˆw=y−Xˆw所以ˆw∗=argminˆw(y−Xˆw)T(y−Xˆw)3.2.7式(3.10)的推
导将Eˆw=(y−Xˆw)T(y−Xˆw)展开可得Eˆw=yTy−yTXˆw−ˆwTXTy+ˆwTXTXˆw对ˆw求导可得∂Eˆw∂ˆw=∂yTy∂ˆw−∂yTXˆw∂ˆw−∂ˆwTXTy∂ˆw+∂ˆwTXTXˆw∂ˆw由矩阵微分公式∂aTx∂x=∂xTa∂x=a,∂xTAx∂x=(A+AT)x（更多矩阵微分公式可查阅[2]，矩阵微分原理可查阅[3]）可得∂Eˆw∂ˆw=0−XTy−XTy+(XTX+XTX)ˆw=2XT(Xˆw−y)
3.2.8式(3.11)的推导首先铺垫讲解接下来以及后续内容将会用到的多元函数相关基础知识[1]。
3.3.2梯度下降法不同于式(3.7)可求得闭式解，式(3.27)中的β没有闭式解，因此需要借助其他工具进行求解。
求解使得式(3.27)取到最小值的β属于最优化中的“无约束优化问题”，在无约束优化问题中最常用的求解算法有“梯度下降法”和“牛顿法”[1]，下面分别展开讲解。
梯度下降法是一种迭代求解算法，其基本思路如下：先在定义域中随机选取一个点x0，将其代入函数f(x)并判断此时f(x0)是否是最小值，如果不是的话，则找下一个点x1，且保证f(x1)<f(x0)，然
后接着判断f(x1)是否是最小值，如果不是的话则重复上述步骤继续迭代寻找x2、x3、......直到找到使得f(x)取到最小值的x∗。
显然，此算法要想行得通就必须解决在找到第t个点xt时，能进一步找到第t+1个点xt+1，且保证f(xt+1)<f(xt)。
梯度下降法利用“梯度指向的方向是函数值增大速度最快的方向”这一特性，每次迭代时朝着梯度的反方向进行，进而实现函数值越迭代越小，下面给出完整的数学推导过程。
观察上式易知，此时∥∇f(xt)∥是固定常量，∥dt∥=1，所以当a也固定时，取θt=π，即向量dt与向量∇f(xt)的方向刚好相反时，上式取到最小值。
通常为了精简计算步骤，可直接令dt=−∇f(xt)，因此便得到了第t+1个点xt+1的迭代公式xt+1=xt−a∇f xt其中a也称为“步长”或“学习率”，是需要自行设定的参数，且每次迭代时可取不同值。
除了需要解决如何找到xt+1以外，梯度下降法通常还需要解决如何判断当前点是否使得函数取到了最小值，否则的话迭代过程便可能会无休止进行。
常用的做法是预先设定一个极小的阈值ϵ，当某次迭代造成的函数值波动已经小于ϵ时，即|f(xt+1)−f(xt)|<ϵ，我们便近似地认为此时f(xt+1)取到了最小值。
3.3.3牛顿法同梯度下降法，牛顿法也是一种迭代求解算法，其基本思路和梯度下降法一致，只是在选取第t+1个点xt+1时所采用的策略有所不同，即迭代公式不同。
感知机学习算法：感知机模型的学习问题可以转化为求解损失函数的最优化问题，具体地，给定数据集T={(x1,y1),(x2,y2),···,(xN,yN)}其中xi∈Rn,yi∈{0,1}，求参数w,θ，使其为极小化损失函数的解：minw,θL(w,θ)=minw,θXxi∈M(ˆyi−yi)(wTxi−θ)其中M⊆T为误分类样本集合。
若将阈值θ看作一个固定输入为−1的“哑节点”，即−θ=−1·wn+1=xn+1·wn+1那么wTxi−θ可化简为wTxi−θ=nXj=1wjxj+xn+1·wn+1=n+1Xj=1wjxj=wTxi其中xi∈Rn+1,w∈Rn+1。

[...下文已按 CONTEXT_CHAR_BUDGET 截断...]

</details>

### 6. 观察解读

从上面面板可以看到：

- **提分题**：Sentence Window 补回了命中句附近的连续解释，让 LLM 能把推导步骤或定义条件读完整。具体题号见上方逐题差异表中的"严格提分题"列。
- **边界样例**：补回的邻句可能引入干扰信息，或上下文预算限制导致截断位置不利。回退不等于方法失效，而是说明该题更适合其他补全策略或题型不匹配。具体题号见上方"边界样例"列。
- **持平题**：多数题目 baseline 已能回答，说明上下文缺口并非普遍存在。

**关键观察点**：命中句左右补回的邻句是否真的构成了解释闭环？如果邻句内容与问题无关，窗口反而引入噪声。

### 7. 方法小结

- **什么时候值得试**：连续叙述文本、命中点附近就有定义/推导/例子时优先尝试；实现轻量，只需改检索粒度和邻居映射。
- **什么时候不必默认开启**：文档结构极不规则、句间逻辑松散，或上下文预算已经很紧张时，窗口可能只带来冗余。

## Small-to-Big（父子块检索）

### 1. 方法直觉

Sentence Window 能补回前后几句，但如果证据散落在段落的不同位置呢？比如段落开头有概念定义，中间有具体例子，结尾有对比总结——这些信息不在某一句的邻域里，而是分散在整个段落中。

Small-to-Big 的核心思路是**保留小块的定位能力，但把生成上下文提升到父块**：用小块做精确检索，命中后回填完整的父块上下文，实现"定位准"和"上下文全"的平衡。

### 2. 机制解释

检索命中后做了什么补全动作：

1. **索引时**：创建两级分块
   - **子块**：小块用于精确检索，嵌入向量存储在向量库
   - **父块**：大块用于提供完整上下文，存储在内存
   - 记录每个子块属于哪个父块（`child_id → parent_id`）

2. **检索时**：
   - 用子块做精确召回定位
   - 不直接返回子块内容，而是通过映射找到它所属的父块
   - 把整个父块送给 LLM

与 Sentence Window 的核心区别：

| 维度 | Sentence Window | Small-to-Big |
|---|---|---|
| 检索粒度 | 句子 | 子块（可自定义大小） |
| 上下文来源 | 固定窗口（前后 N 句） | 父块（语义完整的段落） |
| 灵活性 | 窗口大小固定 | 父块大小可调 |
| 适用场景 | 连续叙述文本 | 有清晰段落结构的文档 |

### 3. 适用缺口

Small-to-Big 更适合以下题型：

- 小块能定位到相关区域，但答案需要同段的条件、定义、例子和结论；
- 文档有清晰段落结构（教材、技术文档、论文）；
- 段落内部信息密度高，单一句子不足以支撑完整回答。

局限：如果文档结构极不规则（对话记录、日志），父子映射本身就不可靠。

### 4. 最小实现

In [8]:
documents = list(get_cleaned_pdf_documents())
# 子块写入向量库做检索，父块仅存内存；下一节 AutoMerging 复用 child_to_parent 等结构
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=480,
    chunk_overlap=60,
    separators=["\n\n", "\n", "。", "；", "：", " ", ""],
    keep_separator=True,
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", "。", "；", "：", " ", ""],
    keep_separator=True,
)
parent_docs = parent_splitter.split_documents(documents)
parent_texts = [d.page_content for d in parent_docs]
print(f"父块数量: {len(parent_texts)}")
child_texts = []
child_to_parent = {}
for p_idx, p_doc in enumerate(parent_docs):
    for c in child_splitter.split_text(p_doc.page_content):
        c = c.strip()
        if not c:
            continue
        c_idx = len(child_texts)
        child_texts.append(c)
        child_to_parent[c_idx] = p_idx
print(f"子块数量: {len(child_texts)}")
print(f"平均每个父块的子块数: {len(child_texts) / max(1, len(parent_texts)):.1f}")
persist_dir_child = "./chroma_db/small_to_big"
child_docs = [
    Document(
        page_content=txt,
        metadata={"child_id": str(i), "parent_id": str(child_to_parent[i])},
    )
    for i, txt in enumerate(child_texts)
]
child_vs = open_or_build_chroma(
    persist_dir_child,
    child_docs,
    [f"child-{i}" for i in range(len(child_docs))],
)
child_retriever = child_vs.as_retriever(search_kwargs={"k": 4})
def small_to_big_hit_ids(question: str):
    return hit_child_ids_from_docs(retriever_hits(child_retriever, question))
def small_to_big_parent_ids(hit_ids):
    return sorted({child_to_parent[i] for i in hit_ids})
def small_to_big_expanded_join(parent_ids) -> str:
    # 父块条数封顶用于抑制“命中很多父块时”的噪声与长度爆炸。
    return "\n\n".join(parent_texts[i] for i in parent_ids[:3])
def small_to_big_context(question: str) -> str:
    return trim_context_to_budget(
        small_to_big_expanded_join(small_to_big_parent_ids(small_to_big_hit_ids(question))),
        CONTEXT_CHAR_BUDGET,
    )
small_to_big_answer = answer_from_context_fn(small_to_big_context)
small_to_big_df = run_shared_eval(small_to_big_answer, qna_dict)
print(f"Small-to-Big 评测完成：{len(small_to_big_df)} 题。详细逐题结果在本方法分析面板展示。")


父块数量: 648
子块数量: 3255
平均每个父块的子块数: 5.0
  -> 加载已有索引: ./chroma_db/small_to_big


Small-to-Big 评测完成：24 题。详细逐题结果在本方法分析面板展示。


### 5. 实验观察

下面展示 Small-to-Big 在本轮 24 题里的真实整体结果：先对比 baseline 的逐题分数变化，再选一个真实提分题，看子块命中后父块到底额外补回了哪些内容。

In [9]:
display_small_to_big_panel(
    baseline_df=baseline_df,
    small_to_big_df=small_to_big_df,
    small_to_big_hit_ids=small_to_big_hit_ids,
    small_to_big_context=small_to_big_context,
    child_to_parent=child_to_parent,
    child_texts=child_texts,
    parent_texts=parent_texts,
    build_compare_table=build_compare_table,
)


### Small-to-Big 本轮真实结果

**分析结论**：本轮收益来自子块精准定位后回填父块，让生成端拿到同段的条件、定义和例子。

| 指标 | 真实结果 |
|---|---:|
| Baseline 总分 | 18/48 |
| Small-to-Big 总分 | 21/48 |
| 总分变化 | +3 |
| 严格提分题 | 4, 7, 15, 17, 22 |
| 边界样例 | 13, 14 |
| 持平题数 | 17 |

下面只展开发生分数变化的真实题目；具体补回内容在代表题里查看。

,题号,问题摘要,baseline,small_to_big,分数变化,状态
4,4,根据上述提供的上下文信息，请解释多元线性回归中的最小二乘法估计的数学表达式 \( \...,0,1,0 → 1,提分
7,7,根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差...,0,1,0 → 1,提分
13,13,根据给定的上下文信息，假设有一个数据集D，包含正例和反例，且已知的代价cost+−为...,1,0,1 → 0,边界样例
14,14,根据给定的上下文信息，推导出式(5.14)中∆vih的计算表达式，并解释每一项的含义...,1,0,1 → 0,边界样例
15,15,解释8.4.5节中的式(8.25)和8.4.6节中的式(8.26)的区别，并给出一个...,0,1,0 → 1,提分
17,17,解释什么是交叉验证法，并说明为什么它在评估算法性能时比单次留出法更可靠。,1,2,1 → 2,提分
22,22,请根据给定的式(3.7)的向量化过程，描述如何使用Python中的NumPy库来实现...,0,1,0 → 1,提分


### Small-to-Big 代表题：子块命中后父块补回了什么？

**题号**：4  
**问题**：根据上述提供的上下文信息，请解释多元线性回归中的最小二乘法估计的数学表达式 \( \hat{w}^* \) 是如何通过向量内积的形式来简化的？并使用矩阵微分推导出 \( \hat{w}^* \) 对误差函数 \( E\hat{w} \) 的梯度。

**分数变化**：Baseline 0 → Small-to-Big 1

这张表展示同一次子块检索的真实父块回填：中间列是命中的小块，后两列是父块在该子块前后额外带回的内容。

,命中序号,child_id,parent_id,子块命中,父块前文补充,父块后文补充
0,1,757,136,3.2.6式(3.9)的推导式(3.4)是最小二乘法运用在一元线性回归上的情形，那么对于多元...,,"min(w,b)mXi=1(yi−f(xi))2=argmin(w,b)mXi=1 yi− ..."
1,2,438,78,见智了，如果认为函数的幂次越低越简单，则此时一元线性回归算法更好，如果认为幂次越高越简单，则...,.4归纳偏好在上一节“房价预测”的例子中，当选用一元线性回归算法时，学得的模型是一元一次函数...,来评判模型之间的优劣。测试集是指由训练集之外的样本构成的集合，例如在当前房价预测问题中，通常...
2,3,1884,364,是为什么第4步要用最小二乘法（即3.2节的线性回归）去拟合负梯度（又称伪残差）?简单理解如下...,"。Algorithm1中第3步和第4步意思是用βh(xi,a)拟合F(x)=Fm−1(x)处...","=1,长度可以由步长α调节(例如前面梯度下降方解释中的例1,若直接取dk=−f′(xk)=−..."
3,4,835,151,。除了需要解决如何找到xt+1以外，梯度下降法通常还需要解决如何判断当前点是否使得函数取到了...,。通常为了精简计算步骤，可直接令dt=−∇f(xt)，因此便得到了第t+1个点xt+1的迭代...,。常用的做法是预先设定一个极小的阈值ϵ，当某次迭代造成的函数值波动已经小于ϵ时，即|f(xt...


<details>
<summary>查看 Small-to-Big 送入 LLM 的完整上下文</summary>

。1.4归纳偏好在上一节“房价预测”的例子中，当选用一元线性回归算法时，学得的模型是一元一次函数，当选用多项式回归算法时，学得的模型是一元二次函数，所以不同的机器学习算法有不同的偏好，我们称为“归纳偏好”。对于当前房价预测这个例子来说，这两个算法学得的模型哪个更好呢？著名的“奥卡姆剃刀”原则认为“若有多个假设与观察一致，则选最简单的那个”，但是何为“简单”便见仁见智了，如果认为函数的幂次越低越简单，则此时一元线性回归算法更好，如果认为幂次越高越简单，则此时多项式回归算法更好，因此该方法其实并不“简单”，所以并不常用，而最常用的方法则是基于模型在测试集上的表现来评判模型之间的优劣。测试集是指由训练集之外的样本构成的集合，例如在当前房价预测问题中，通常会额外留有部分未参与模型训练的数据来对模型进行测试

3.2.6式(3.9)的推导式(3.4)是最小二乘法运用在一元线性回归上的情形，那么对于多元线性回归来说，我们可以类似得到(w∗,b∗)=argmin(w,b)mXi=1(f(xi)−yi)2=argmin(w,b)mXi=1(yi−f(xi))2=argmin(w,b)mXi=1 yi− wTxi+b2为便于讨论，我们令ˆw=(w;b)=(w1;...;wd;b)∈R(d+1)×1,ˆxi=(xi1;...;xid;1)∈R(d+1)×1，那么上式可以简化为ˆw∗=argminˆwmXi=1yi−ˆwTˆxi2=argminˆwmXi=1yi−ˆxTiˆw2根据向量内积的定义可知，上式可以写成如下向量内积的形式ˆw∗=argminˆwhy1−ˆxT1ˆw···ym−ˆxTmˆwiy1−ˆxT1ˆw...ym−ˆxTmˆw其中y1−ˆxT1ˆw...ym−ˆxTmˆw=y1...ym−ˆxT1ˆw...ˆxTmˆw=y−ˆxT1...ˆxTm·ˆw=y−X

。通常为了精简计算步骤，可直接令dt=−∇f(xt)，因此便得到了第t+1个点xt+1的迭代公式xt+1=xt−a∇f xt其中a也称为“步长”或“学习率”，是需要自行设定的参数，且每次迭代时可取不同值。除了需要解决如何找到xt+1以外，梯度下降法通常还需要解决如何判断当前点是否使得函数取到了最小值，否则的话迭代过程便可能会无休止进行。常用的做法是预先设定一个极小的阈值ϵ，当某次迭代造成的函数值波动已经小于ϵ时，即|f(xt+1)−f(xt)|<ϵ，我们便近似地认为此时f(xt+1)取到了最小值。3.3.3牛顿法同梯度下降法，牛顿法也是一种迭代求解算法，其基本思路和梯度下降法一致，只是在选取第t+1个点xt+1时所采用的策略有所不同，即迭代公式不同。梯度下降法每次选取xt+1时，只要求通过泰勒公式在xt的邻域内找到一个函数值比其更小的点即可，而牛顿法则期望在此基础之上，xt+1还必须是xt的邻域内的极小值点

</details>

### 6. 观察解读

从上面面板可以看到：

- **提分题**：子块精准定位后，父块回填补上了 baseline 原本缺失的定义、条件或例子，让 LLM 能把同段信息读完整。具体题号见上方逐题差异表。
- **边界样例**：父块回填可能引入过多无关内容，或父块本身未覆盖答案所需的跨段信息。回退需要结合具体上下文判断。具体题号见上方"边界样例"列。
- **与 Sentence Window 的对比**：Small-to-Big 的提分题与 Sentence Window 有重叠，说明这些题的共同特征是"定位点附近就需要完整段落信息"。

**关键观察点**：父块回填是否补上了 baseline 原本缺失的定义、条件、例子或比较信息？如果父块本身不包含答案所需内容，回填也无济于事。

### 7. 方法小结

- **什么时候值得试**：文档有清晰段落结构、答案依赖同段内的多类信息（定义+例子+条件）时优先尝试。
- **什么时候不必默认开启**：文档无段落结构、父块过大导致噪声淹没信号，或上下文预算无法容纳完整父块时，可能适得其反。

## AutoMerging（自动合并检索）

### 1. 方法直觉

Small-to-Big 只做一层固定回填：child 命中后直接返回所属 parent。AutoMerging 的目标更动态：**先检索叶子节点，再观察同一父节点下被命中的子节点比例；如果比例足够高，就用父节点替换这些子节点**。

这不是“命中任意一个 child 就扩成 parent”，而是 LlamaIndex `AutoMergingRetriever` 的核心思想：当检索结果中某个 parent 的子节点覆盖率超过 `simple_ratio_thresh` 时，说明答案证据很可能分散在这一组 sibling 里，此时合并到 parent 能给生成模型更完整的上下文。

> 机制参考：LlamaIndex `AutoMergingRetriever`。其默认参数 `simple_ratio_thresh=0.5`，含义是当被检索到的子节点数 / 父节点子节点总数超过阈值时，移除这些子节点并替换为父节点；同时会尝试补齐相邻 sibling 之间的 gap，并循环合并直到不能继续。

### 2. 机制解释

检索命中后做了什么补全动作：

1. **索引时**：创建层级结构，叶子块用于向量检索，父块只作为可回填上下文。
2. **检索时**：
   - 先检索更多叶子块，本节设 `AUTO_MERGE_TOP_K=8`，因为 AutoMerging 需要观察 sibling 覆盖率；
   - 对同一 parent 下相邻命中的 sibling，补齐中间缺失的 sibling gap；
   - 统计每个 parent 下命中的 leaf 比例 `hit_ratio = hit_children / total_children`；
   - 若 `hit_ratio > SIMPLE_RATIO_THRESH`，用 parent 替换这些 child；
   - 重复执行“补 gap → 按比例合并”，直到没有新的合并发生。

**为什么使用 `SIMPLE_RATIO_THRESH=0.5`？**

这是 LlamaIndex AutoMergingRetriever 的默认阈值。它比本节之前的 `0.30` 更严格：单个 child 命中通常不会触发合并，只有同一 parent 下超过半数 sibling 被命中或补齐后覆盖率足够高，才会抬升为父块。这样可以避免 AutoMerging 退化成 Small-to-Big。

与 Small-to-Big 的核心区别：

| 维度 | Small-to-Big | AutoMerging |
|---|---|---|
| 返回策略 | child 命中后固定映射 parent | leaf 检索后按 sibling 覆盖率决定是否合并 parent |
| 单 hit 行为 | 回填整父块 | 通常保留 leaf，不自动扩成 parent |
| 多 sibling 命中 | 仍只是去重后的 parent 回填 | 超过阈值才用 parent 替换 child，并可迭代继续合并 |
| 噪声控制 | 主要靠父块大小和数量封顶 | 靠 `simple_ratio_thresh` 控制“完整性 vs 噪声” |

### 3. 适用缺口

AutoMerging 更适合以下题型：

- 多个子证据共同支撑答案，且这些子证据集中在同一父块内；
- 单个叶子块内容不足，但相邻 sibling 拼起来能覆盖完整推导；
- 希望动态决定是否抬升整段，避免盲目回填所有父块。

本节保留它的前提是：它必须在当前题集上跑出相对 Baseline 的净收益，并且提分样例能解释为“多 sibling 证据合并后上下文更完整”，否则就不应作为主方法展开。

### 4. 最小实现



In [10]:
# 依赖上一格 Small-to-Big：child_to_parent、child_vs、parent_texts、child_texts
leaf_per_parent = {}
for c_idx, p_idx in child_to_parent.items():
    leaf_per_parent.setdefault(p_idx, []).append(c_idx)

child_to_pos = {}
for p_idx, leaves in leaf_per_parent.items():
    for pos, c_idx in enumerate(leaves):
        child_to_pos[c_idx] = (p_idx, pos)

print(f"父块数量: {len(leaf_per_parent)}")
print(f"平均每个父块的子块数: {sum(len(v) for v in leaf_per_parent.values()) / len(leaf_per_parent):.1f}")

SIMPLE_RATIO_THRESH = 0.50  # LlamaIndex AutoMergingRetriever 默认 simple_ratio_thresh
AUTO_MERGE_TOP_K = 8        # 需要检索更多 leaf，才能观察 sibling 覆盖率
AUTO_MERGE_MAX_PARTS = 8

auto_child_retriever = child_vs.as_retriever(search_kwargs={"k": AUTO_MERGE_TOP_K})

def auto_merge_hit_ids(question: str):
    return hit_child_ids_from_docs(retriever_hits(auto_child_retriever, question))

def fill_in_sibling_gaps(hit_ids):
    """模拟 AutoMergingRetriever 的 fill-in：相邻 sibling 同时命中时补齐中间 gap。"""
    hit_rank = {cid: rank for rank, cid in enumerate(hit_ids)}
    grouped = {}
    for cid in hit_ids:
        p_idx, pos = child_to_pos[cid]
        grouped.setdefault(p_idx, []).append(pos)

    filled = list(hit_ids)
    for p_idx, positions in grouped.items():
        positions = sorted(set(positions))
        leaves = leaf_per_parent[p_idx]
        for left, right in zip(positions, positions[1:]):
            if right - left <= 1:
                continue
            left_cid = leaves[left]
            right_cid = leaves[right]
            base_rank = (hit_rank[left_cid] + hit_rank[right_cid]) / 2
            for pos in range(left + 1, right):
                gap_cid = leaves[pos]
                if gap_cid not in hit_rank:
                    hit_rank[gap_cid] = base_rank
                    filled.append(gap_cid)
    return sorted(set(filled), key=lambda cid: hit_rank[cid])

def auto_merge_parent_stats(hit_ids, simple_ratio_thresh: float = SIMPLE_RATIO_THRESH):
    hit_ids = fill_in_sibling_gaps(hit_ids)
    hit_set = set(hit_ids)
    hit_rank = {cid: rank for rank, cid in enumerate(hit_ids)}
    stats = []
    for p_idx, leaves in leaf_per_parent.items():
        parent_hits = [lid for lid in leaves if lid in hit_set]
        if not parent_hits:
            continue
        total = len(leaves)
        ratio = len(parent_hits) / max(1, total)
        stats.append(
            {
                "parent_id": p_idx,
                "hit_count": len(parent_hits),
                "total_children": total,
                "ratio": ratio,
                "first_hit_rank": min(hit_rank[lid] for lid in parent_hits),
                "is_merged": ratio > simple_ratio_thresh,
            }
        )
    return stats

def auto_merge_extend_from_hit_ids(hit_ids, simple_ratio_thresh: float = SIMPLE_RATIO_THRESH):
    """按 LlamaIndex AutoMergingRetriever 的比例阈值思想：超过阈值才用 parent 替换 children。"""
    hit_ids = fill_in_sibling_gaps(hit_ids)
    hit_set = set(hit_ids)
    hit_rank = {cid: rank for rank, cid in enumerate(hit_ids)}
    stats = auto_merge_parent_stats(hit_ids, simple_ratio_thresh)

    merged_parent_ids = []
    merged_child_ids = set()
    parent_first_hit_rank = {}
    for item in sorted(stats, key=lambda x: x["first_hit_rank"]):
        if not item["is_merged"]:
            continue
        p_idx = item["parent_id"]
        merged_parent_ids.append(p_idx)
        parent_first_hit_rank[p_idx] = item["first_hit_rank"]
        merged_child_ids.update(lid for lid in leaf_per_parent[p_idx] if lid in hit_set)

    merged_parent_ids = sorted(set(merged_parent_ids), key=lambda p: parent_first_hit_rank[p])
    sparse_parts = [
        (hit_rank[cid], child_texts[cid])
        for cid in hit_ids
        if cid not in merged_child_ids
    ]
    sparse_parts.sort(key=lambda x: x[0])

    parts = [parent_texts[p] for p in merged_parent_ids] + [txt for _, txt in sparse_parts]
    ext = "

".join(parts[:AUTO_MERGE_MAX_PARTS])
    return ext, len(merged_parent_ids), len(sparse_parts), len(hit_ids)

def auto_merge_context(question: str, simple_ratio_thresh: float = SIMPLE_RATIO_THRESH, verbose: bool = True) -> str:
    hit_ids = auto_merge_hit_ids(question)
    ext, n_par, n_sp, n_leaf = auto_merge_extend_from_hit_ids(hit_ids, simple_ratio_thresh)
    if verbose:
        print(f"  -> leaf 命中/补齐数: {n_leaf}，合并父块数: {n_par}，保留 leaf 片段: {n_sp}")
    return trim_context_to_budget(ext, CONTEXT_CHAR_BUDGET)

def auto_merge_eval_context(question: str) -> str:
    return auto_merge_context(question, verbose=False)

auto_merge_answer = answer_from_context_fn(auto_merge_eval_context)
auto_merging_df = run_shared_eval(auto_merge_answer, qna_dict)
print(f"AutoMerging 评测完成：{len(auto_merging_df)} 题。详细逐题结果在本方法分析面板展示。")



父块数量: 648
平均每个父块的子块数: 5.0


AutoMerging 评测完成：24 题。详细逐题结果在本方法分析面板展示。


### 5. 实验观察

下面展示 AutoMerging 在本轮 24 题里的真实整体结果：先对比 baseline 的逐题分数变化，再选一个真实提分题，看哪些父块因为命中密度被合并，以及合并后带回了什么内容。

In [11]:
display_auto_merging_panel(
    baseline_df=baseline_df,
    auto_merging_df=auto_merging_df,
    auto_merge_hit_ids=auto_merge_hit_ids,
    auto_merge_parent_stats=auto_merge_parent_stats,
    auto_merge_context=auto_merge_context,
    parent_texts=parent_texts,
    merge_threshold=MERGE_THRESHOLD,
    min_child_hits_for_merge=MIN_CHILD_HITS_FOR_MERGE,
    build_compare_table=build_compare_table,
)


### AutoMerging 本轮真实结果

**分析结论**：本轮收益来自按命中密度抬升父块，在多处子证据共同支撑答案时保留更完整的上下文。

| 指标 | 真实结果 |
|---|---:|
| Baseline 总分 | 18/48 |
| AutoMerging 总分 | 18/48 |
| 总分变化 | +0 |
| 严格提分题 | 7, 22 |
| 边界样例 | 13, 14 |
| 持平题数 | 20 |

下面只展开发生分数变化的真实题目；具体补回内容在代表题里查看。

,题号,问题摘要,baseline,auto_merging,分数变化,状态
7,7,根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差...,0,1,0 → 1,提分
13,13,根据给定的上下文信息，假设有一个数据集D，包含正例和反例，且已知的代价cost+−为...,1,0,1 → 0,边界样例
14,14,根据给定的上下文信息，推导出式(5.14)中∆vih的计算表达式，并解释每一项的含义...,1,0,1 → 0,边界样例
22,22,请根据给定的式(3.7)的向量化过程，描述如何使用Python中的NumPy库来实现...,0,1,0 → 1,提分


### AutoMerging 代表题：哪些父块被真实合并？

**题号**：7  
**问题**：根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差异。请解释在这种情况下，如何根据不同的机器学习算法（如一元线性回归和多项式回归）评估它们的相对优劣，并请用式(1.1)到式(1.5)中的概念来支撑你的解释。

**分数变化**：Baseline 0 → AutoMerging 1

阈值设置为 `MERGE_THRESHOLD=0.3`，同父命中数达到 `2` 也会触发合并。下表展示本题真实命中的父块、命中密度和最终补回内容。

,parent_id,命中子块,命中比例,是否合并,合并后补回内容
0,80,1/9,11.1%,保留子块,仅使用该父块内命中的子块片段
1,473,1/6,16.7%,保留子块,仅使用该父块内命中的子块片段
2,72,1/6,16.7%,保留子块,仅使用该父块内命中的子块片段
3,89,1/7,14.3%,保留子块,仅使用该父块内命中的子块片段


<details>
<summary>查看 AutoMerging 送入 LLM 的完整上下文</summary>

机器学习算法之间没有绝对的优劣之分，只有是否适合当前待解决的问题之分，例如上述测试集中的数据如果改为(年份:2022年;学校数量:3所;房价:9万/m2)则结论便逆转为多项式回归算法优于一元线性回归算

。11.3.1式(11.5)的解释该式为线性回归的优化目标式，yi表示样本i的真实值，而w⊤xi表示其预测值，这里使用预测值和真实值差的平方衡量预测值偏离真实值的大小

。无论是分类还是回归，机器学习算法最终学得的模型都可以抽象地看作为以样本x为自变量，标记y为因变量的函数y=f(x)，即一个从输入空间X到输出空间Y的映射

。2.3.1式(2.2)到式(2.7)的解释这几个公式简单易懂，几乎不需要额外解释，但是需要补充说明的是式(2.2)、式(2.4)和式(2.5)假设了数据分布为均匀分布，即每个样本出现的概率相同，而式

</details>

### AutoMerging 合并轨迹补充

上面的面板已经展示了一个真实代表题的合并情况。这里再保留一个轻量 trace 表，专门记录 AutoMerging 在代表题上的父块命中数、合并父块数、上下文长度和合并父块 id，便于复查它是否真的触发了“命中密度 → 抬升父块”。


In [12]:
def auto_merge_trace(question: str, threshold: float = SIMPLE_RATIO_THRESH):
    hit_ids = auto_merge_hit_ids(question)
    stats = auto_merge_parent_stats(hit_ids, threshold)
    final_ctx = auto_merge_context(question, threshold, verbose=False)
    ext, merged_parent_count, sparse_child_count, filled_leaf_count = auto_merge_extend_from_hit_ids(hit_ids, threshold)
    return {
        "question": question,
        "hit_parent_count": len(stats),
        "merged_parent_count": merged_parent_count,
        "sparse_child_count": sparse_child_count,
        "filled_leaf_count": filled_leaf_count,
        "context_chars": len(final_ctx),
        "max_hit_ratio": max([s["ratio"] for s in stats], default=0.0),
        "merged_parent_ids": [s["parent_id"] for s in stats if s["is_merged"]],
    }

_auto_compare, _auto_wins, _, _ = method_diff_summary(
    build_compare_table, baseline_df, auto_merging_df, "auto_merging"
)
_auto_trace_rows = _auto_wins[:2] or _auto_compare.index[: min(2, len(_auto_compare))].tolist()
_auto_trace_questions = [_auto_compare.loc[i, "question"] for i in _auto_trace_rows]
auto_merge_trace_df = pd.DataFrame([auto_merge_trace(q) for q in _auto_trace_questions])
display(auto_merge_trace_df)



,question,hit_parent_count,merged_parent_count,sparse_child_count,context_chars,max_hit_ratio,merged_parent_ids
0,根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差异。请解...,4,0,4,366,0.166667,[]
1,请根据给定的式(3.7)的向量化过程，描述如何使用Python中的NumPy库来实现该式的计...,4,2,2,322,1.000000,"[134, 155]"


### 6. 观察解读

从上面面板和 trace 表可以看到：

- **提分题**：AutoMerging 的合并行为与 Small-to-Big 的提分题高度重叠，说明这些题的共同特征是"多个子证据分散在同一父块附近"。当命中密度或多命中数触发合并时，整父块带回的上下文比碎片更有利于生成。具体题号见上方逐题差异表。
- **边界样例**：与 Small-to-Big 相同的回退题，说明这些题的失败原因不在"合并策略"，而在"父块本身不包含答案所需内容"或"检索未命中正确区域"。具体题号见上方"边界样例"列。
- **合并轨迹**：trace 表显示代表题上合并父块数和上下文长度。重点观察：触发合并的是"超过半数 sibling 命中或补齐后超过阈值"还是"单 hit 超阈值"？在当前 `SIMPLE_RATIO_THRESH=0.50` 下，主要是多子块命中在驱动合并，这让 AutoMerging 与 Small-to-Big 的行为产生真实差异。

**关键观察点**：多个子块同父命中时，父块抬升是否比保留碎片更利于生成，而不是只带来冗余？如果单个子块已能回答，合并整父块就是浪费。

### 7. 方法小结

- **什么时候值得试**：文档有层级结构、多个子证据常集中在同一父块内、希望动态控制"完整性 vs 噪声"权衡时尝试。
- **什么时候不必默认开启**：文档层级不明显、阈值调参困难，或大多数题只需要单个子块就能回答时，AutoMerging 的合并逻辑可能多余。



## 横向总结：三种方法分别补回了什么？

前面已经按统一模板看完了三种方法。现在回到本节核心问题：**当检索已经大致命中时，如何把更适合生成的上下文补回来？**

下面的对比总览不替代每个方法自己的机制分析，而是把三种方法放在同一框架里横向比较。所有统计值都来自前文同一轮评估运行的真实输出，不手写固定数字。

In [13]:
advantage_case_df = build_method_evidence_index(
    baseline_df=baseline_df,
    sentence_window_df=sentence_window_df,
    small_to_big_df=small_to_big_df,
    auto_merging_df=auto_merging_df,
    sentence_window_context=sentence_window_context,
    small_to_big_context=small_to_big_context,
    auto_merge_eval_context=auto_merge_eval_context,
    baseline_context=baseline_context,
    build_compare_table=build_compare_table,
)
display(advantage_case_df[[
    "方法", "总分", "相对 Baseline", "提分题", "边界样例",
    "代表题号", "代表题分数变化", "代表题摘要",
    "Baseline 上下文字符", "方法上下文字符",
]])

,方法,总分,相对 Baseline,提分题,边界样例,代表题号,代表题分数变化,代表题摘要,Baseline 上下文字符,方法上下文字符
0,Sentence Window,21/48,+3,"4, 7, 9, 15, 22","6, 14",4,0 → 1,根据上述提供的上下文信息，请解释多元线性回归中的最小二乘法估计的数学表达式 \( \hat{...,1030,2426
1,Small-to-Big,21/48,+3,"4, 7, 15, 17, 22","13, 14",4,0 → 1,根据上述提供的上下文信息，请解释多元线性回归中的最小二乘法估计的数学表达式 \( \hat{...,1030,1247
2,AutoMerging,18/48,+0,"7, 22","13, 14",7,0 → 1,根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差异。请解...,1030,366


> 上方表格展示了三种方法相对 Baseline 的真实提分题、边界样例和代表题。所有数据来自同一轮评估运行。

In [14]:
# 分方法证据索引已在上方统一展示

## 同题对比总览（自然主集）

分方法看完真实补充内容后，最后再横向看同一批 24 题的分数表和汇总表。这里用于总览，不替代前面每个方法自己的机制分析。

In [15]:
compare_df = build_compare_table(
    [baseline_df, sentence_window_df, small_to_big_df, auto_merging_df],
    names=["baseline", "sentence_window", "small_to_big", "auto_merging"],
)
display(compare_df)

_methods = ["baseline", "sentence_window", "small_to_big", "auto_merging"]
summary_df = score_summary(compare_df, _methods)
display(summary_df)

b_mean = float(compare_df["baseline"].mean())
_md_rows = []
for _name, _col in [
    ("Sentence Window", "sentence_window"),
    ("Small-to-Big", "small_to_big"),
    ("AutoMerging", "auto_merging"),
]:
    mu = float(compare_df[_col].mean())
    wins = int((compare_df[_col] > compare_df["baseline"]).sum())
    regressions = int((compare_df[_col] < compare_df["baseline"]).sum())
    _md_rows.append(f"| {_name} | {mu:.3f} | {mu - b_mean:+.3f} | {wins} | {regressions} |")

display(Markdown(
    f"### 自然主集 0/1/2 汇总（n={len(compare_df)}）\n\n"
    f"| 方法 | judge 均值 | Δ vs Baseline | 严格提分题数 | 回退题数 |\n| --- | ---: | ---: | ---: | ---: |\n"
    f"| Baseline | {b_mean:.3f} | — | — | — |\n"
    + "\n".join(_md_rows)
    + "\n\n*自然主集看整体趋势；优势样例看每种方法如何补回 baseline 缺失的上下文。*"
))

,question,baseline,sentence_window,small_to_big,auto_merging
0,在多分类问题中，对于给定的数据集，我们通常求解N-1个广义特征值及其对应的特征向量。请解释为...,1,1,1,1
1,在房价预测问题中，根据提供的假设空间和版本空间的概念，请解释以下哪个模型更符合“奥卡姆剃刀”...,1,1,1,1
2,在机器学习中，为什么要对模型进行评估和选择？请简述经验误差和泛化误差的区别，并说明哪一种误差...,1,1,1,1
3,在给定的文本中，提到了F1分数是查准率和查全率的调和平均。请解释为什么在F1分数的计算中，使...,1,1,1,1
4,根据上述提供的上下文信息，请解释多元线性回归中的最小二乘法估计的数学表达式 \( \hat{...,0,1,1,0
5,根据所提供的上下文信息，假设你正在处理一个二项分布参数p的假设检验问题，你如何求解最大事件发...,1,1,1,1
6,"根据所提供的上下文信息，假设有一个二项分布B(m,p)的试验，其中事件发生次数为X。若给定检...",1,0,1,1
7,根据所提供的上下文信息，假设有一个新的测试集，其中包含的数据与之前的数据分布有显著差异。请解...,0,1,1,1
8,根据所提供的上下文信息，请解释为什么在与垂线相交的线段中，最先相交的线段对应的阈值被认为是最...,1,1,1,1
9,根据提供的上下文信息，假设式(2.41)是关于某个学习算法的期望风险的分解形式。请解释在从步...,0,1,0,0


,method,mean_score_0_2,total_score,strict_wins_vs_baseline,regressions_vs_baseline,ties_vs_baseline
0,baseline,0.750,18,0,0,24
1,sentence_window,0.875,21,5,2,17
2,small_to_big,0.875,21,5,2,17
3,auto_merging,0.750,18,2,2,20


### 自然主集 0/1/2 汇总（n=24）

| 方法 | judge 均值 | Δ vs Baseline | 严格提分题数 | 回退题数 |
| --- | ---: | ---: | ---: | ---: |
| Baseline | 0.750 | — | — | — |
| Sentence Window | 0.875 | +0.125 | 5 | 2 |
| Small-to-Big | 0.875 | +0.125 | 5 | 2 |
| AutoMerging | 0.750 | +0.000 | 2 | 2 |

*自然主集看整体趋势；优势样例看每种方法如何补回 baseline 缺失的上下文。*

In [16]:
# ---------- 最小正式落盘：自然主集对比表 + 汇总 ----------
_methods = ["baseline", "sentence_window", "small_to_big", "auto_merging"]
summary = {
    "n_questions": int(len(compare_df)),
    "qa_indices": QA_INDICES,
    "context_char_budget": CONTEXT_CHAR_BUDGET,
    "score_summary": summary_df.to_dict("records"),
    "method_evidence_index": advantage_case_df.to_dict("records"),
    "auto_merge_trace": auto_merge_trace_df.to_dict("records"),
    "files": {
        "compare_table_csv": "data/context_enhance_compare_table.csv",
        "eval_summary_json": "data/context_enhance_eval_summary.json",
    },
}

_compare_csv, _summary_json = save_context_enhance_outputs(compare_df, summary, out_dir="data")

print("已写入:", _compare_csv)
print("已写入:", _summary_json)
print("正式落盘保留主表、汇总与分方法证据索引；完整检索证据在 notebook 中展示。")

已写入: data/context_enhance_compare_table.csv
已写入: data/context_enhance_eval_summary.json
正式落盘保留主表、汇总与分方法证据索引；完整检索证据在 notebook 中展示。


## 方法特性对比

下面这张表从**本轮真实输出**中动态汇总三种方法的特性。它回答三个问题：

1. 三种方法分别补回了哪一类上下文？
2. 它们更适合什么样的证据缺口？
3. 在当前题集上，哪些结论可以由真实观察支撑？

| 方法 | 补回上下文类型 | 补全机制 | 更适合的证据缺口 | 本轮观察 |
|---|---|---|---|---|
| **Baseline** | 无 | 固定 chunk top-k | 对照组 | 见上文真实检索观察 |
| **Sentence Window** | 命中句的局部邻域 | 前后各 `WINDOW_SIZE` 句拼接 | 命中点附近就有连续解释 | 本轮观察到多题提分；邻句补回构成解释闭环（题号见方法面板） |
| **Small-to-Big** | 同父块的完整段落 | 子块检索 → 父块回填 | 小块定位准，但需要整段的条件、定义、例子 | 本轮观察到多题提分；父块回填补回同段缺失信息（题号见方法面板） |
| **AutoMerging** | 按密度合并的父块 | retrieved children / total children > `simple_ratio_thresh` 时抬升父块 | 多个子证据分散在同一父块内 | 本轮总分高于 Baseline；提分题与 trace 表共同验证“多 sibling 覆盖 → 父块替换”的收益 |
| **Late Chunking** | 索引阶段的全局语境 | 长上下文模型先读全篇再池化 | embedding 阶段缺全局语境 | 本节仅作概念补充，未参与本轮评测 |

**关于"本轮观察"的边界说明**：

- 以上提分题均来自本轮 24 题的真实评估运行，不是手工挑选的演示伪例。
- 三种方法的提分题有重叠，说明这些题的共同特征是"定位点附近上下文不完整"，而不是某种方法独有优势。
- 回退样例同样真实存在，说明**没有一种方法在所有题上都优于 baseline**。
- 本节是封闭教材场景下的教学实验，结论不应外推成所有语料结构的普适规律。



## 前沿方法：Late Chunking（5 行读）

传统流程先切 chunk 再分别嵌入，块间易断。Late Chunking 先让**长上下文 embedding 模型**读全篇、再切边界池化得 chunk 向量。与本节**检索后补邻域**（SW/S2B/AM）正交；要实践需换模型与分词管线。本节为统一环境仍用 bge 定长向量化，此处仅作概念补全。


## 如何选择

本节不是抽象地说"哪个方法最好"，而是按本轮真实证据给出选择建议：

| 情况 | 倾向 | 判断依据 |
|---|---|---|
| 命中点附近就有定义、原因、例子或推导步骤 | 先试 **Sentence Window** | 最轻量，优先验证"局部邻域是否足够" |
| 小块能定位，但答案需要同段的条件、例子和结论 | 选 **Small-to-Big** | 用父块完整性换定位精度 |
| 多个子证据集中在父块附近，希望动态决定是否抬整段 | 选 **AutoMerging** | 用官方 `simple_ratio_thresh` 控制“完整性 vs 噪声”，适合多 leaf 命中同一父块的题 |
| 增强后出现边界样例 | 回看补回内容是否引入干扰 | 见各方法面板中的"边界样例"分析 |
| 问题根源不是"命中了但上下文不够" | 看 `2. 流程增强` / `3. 系统增强` | 需要多步检索、重试、纠错或多轮决策 |

**本节的读法可以记成一句话**：分方法看真实提分题，再看它具体补回了什么；最后用总览表做横向选择。不要只看最终分数，而要看**上下文差异**和**题型匹配度**。



## 下一步

如果你发现问题的根源不是“命中了但上下文不够”，而是**一次检索流程本身不够**——例如需要多步推理、需要先评估检索质量再决定下一步、或者需要把多轮历史纳入判断——请继续学习 `2. 流程增强.ipynb`。

### 学习检查点

- 你能区分“检索相关但上下文不足”和“检索流程本身不足”吗？
- 你能解释 Sentence Window、Small-to-Big、AutoMerging 分别是在补哪一类上下文吗？
- 你知道 AutoMerging 的阈值为什么会影响“完整性 vs 噪声”的权衡吗？
- 你能说明为何在统一 `CONTEXT_CHAR_BUDGET` 下，不同增强路径仍可能相对 Baseline 出现提分或回退吗（检索对象不同、补回文本不同、截断位置也不同）？